# transports-publics-france : démonstration

Le package construit, à partir des GTFS disponibles sur [transport.data.gouv.fr](https://transport.data.gouv.fr),
une table des arrêts et des lignes qui les desservent, et un rapport par réseau.

| Fonction | Rôle |
|---|---|
| `run_gtfs_pipeline` | Télécharge et unifie les GTFS, puis construit la table des arrêts × lignes |
| `build_network_report` | Produit un rapport par réseau |

## 0. Installation et configuration

Le package sera bientôt disponible sur PyPI :

```bash
uv add transports-publics-france     # ou : pip install transports-publics-france
```

En attendant, depuis la racine du dépôt : `uv pip install -e .` ou `python -m pip install -e .`

In [1]:
from datetime import date
from pathlib import Path
import polars as pl

from transports_publics_france import run_gtfs_pipeline, build_network_report

RUN_DATE = date.today()   # date de l'instantané : les services actifs sont évalués à cette date
DATA_DIR = Path("data")   # dossier où tout est écrit

## 1. Lancer le pipeline

⚠️ Il porte sur **l'ensemble des GTFS nationaux** : le premier lancement est long et volumineux.

`redownload=True` (défaut) télécharge les archives. \
Avec `False`, rien n'est téléchargé et seules les archives déjà présentes dans `DATA_DIR/<run_date>/` sont traitées.

`run_date=date.today()` (défaut) lance le script pour les données d'aujourd'hui. \
Si l'on veut utiliser des archives plus anciennes, il faut utiliser `run_date = date(YYYY, MM, DD)`

In [3]:
run_gtfs_pipeline(out_dir=DATA_DIR, run_date=RUN_DATE, redownload=True)

Failed to download horaires-theoriques-de-loffre-du-reseau-ferre-regional-de-transport from https://transport.data.gouv.fr/resources/83620/download. Retrying with original_url...
Definitive failure for horaires-theoriques-de-loffre-du-reseau-ferre-regional-de-transport : 403 Client Error: Forbidden for url: https://geocatalogue.hautsdefrance.fr/gtfs/HDF_TER_GTFS_J___90_jours.zip
Failed to download arrets-horaires-et-parcours-theoriques-gtfs-du-reseau-routier-regional-de-transport-scolaire-et-interurbain-59-nord from https://www.data.gouv.fr/api/1/datasets/r/1e2937e3-d495-44bc-aec0-bd5da085e3a4. Retrying with original_url...
Definitive failure for arrets-horaires-et-parcours-theoriques-gtfs-du-reseau-routier-regional-de-transport-scolaire-et-interurbain-59-nord : 403 Client Error: Forbidden for url: https://geocatalogue.hautsdefrance.fr/gtfs/RHDF_GTFS_COM_SCO_59_P4.zip
Failed to download arrets-horaires-et-parcours-theoriques-gtfs-du-reseau-routier-regional-de-transport-scolaire-et-inte

Run date: 2026-10-30 Day of the week: friday


Dataset 12616 : no stops, pass.
Dataset 14652 : no stops, pass.
Dataset 22408 : no active services today, pass.
Dataset 39593 : no active services today, pass.
Dataset 78387 : no stops, pass.
Dataset 79274 : no active services today, pass.
Dataset 79821 : no active services today, pass.
Dataset 80253 : no active services today, pass.
Dataset 80394 : no active services today, pass.
Dataset 81193 : no active services today, pass.
Dataset 81342 : no active services today, pass.
Dataset 81365 : no active services today, pass.
Dataset 81367 : no active services today, pass.
Dataset 81413 : no active services today, pass.
Dataset 81416 : no active services today, pass.
Dataset 81535 : no active services today, pass.
Dataset 81645 : required columns missing in stop_times → stop_id
Dataset 81648 : required columns missing in stop_times → stop_id
Dataset 81649 : required columns missing in stop_times → stop_id
Dataset 81650 : required columns missing in stop_times → stop_id
Dataset 81651 : requ

In [8]:
date_str = RUN_DATE.strftime("%Y%m%d")
rapport_extraction = pl.read_csv(DATA_DIR / f"{date_str}_resultats_extraction.csv")
display(rapport_extraction.head())
display(rapport_extraction['result'].value_counts())

dataset_id,result
i64,str
10230,"""success"""
11492,"""success"""
11681,"""success"""
11725,"""success"""
12601,"""success"""


result,count
str,u32
"""no stops""",5
"""no services""",69
"""missing routes,trips,stop_time…",1
"""failed extract""",2
"""invalid schema""",7
"""success""",462


## 2. La table des arrêts × lignes

Une ligne par arrêt et par ligne qui le dessert à `RUN_DATE`.

In [ ]:

stops = pl.read_parquet(DATA_DIR / f"all_stops_data_{date_str}.parquet")

print(stops.shape)
stops.head()

(273742, 19)


stop_name_red,route_short_name,route_type,stop_id,route_id,agency_id,latitude,longitude,stop_name,route_long_name,agency_name,agency_lang,freq_ppm_max,ligne_ad,ligne_ad_partiel,date_extraction,dataset_id,stop_id_red,route_id_red
str,str,str,str,str,str,f64,f64,str,str,str,str,i64,bool,bool,str,str,str,str
"""tour auvergne""","""85""","""bus""","""80921_IDFM:71373""","""80921_IDFM:C01117""","""80921_IDFM:1090""",48.879365,2.34319,"""La Tour D'Auvergne""","""85""","""Boucles Nord de Seine""",null,13,false,false,"""2026-10-30""","""80921""","""71373""","""C01117"""
"""champlain""","""C4""","""bus""","""81942_FR:76351:ZE:3545:ATOUMOD…","""81942_ATOUMOD003:Line:C4:LOC""","""81942_ATOUMOD003:Network:003:L…",49.495613,0.094591,"""S. De Champlain""","""Grand Hameau - Gares""","""LiA (Le Havre Seine Métropole)""","""FR""",10,false,false,"""2026-10-30""","""81942""","""ATOUMOD003""","""LOC"""
"""porte douai""","""15""","""bus""","""81995_776""","""81995_15""","""81995_ILE""",50.618419,3.072668,"""Porte De Douai""","""Ligne 15""","""ILEVIA""","""FR""",8,false,false,"""2026-10-30""","""81995""","""81995_776""","""81995_15"""
"""centre thouare loire""","""77""","""bus""","""79818_NANTES:StopPlace:FR_NAOL…","""79818_NANTES:Line:77""","""79818_NANTES:Operator:1""",47.266777,-1.441362,"""Centre De Thouaré-Sur-Loire""","""Sables D'Or - Haluchère - Bati…","""Naolib""",null,5,false,false,"""2026-10-30""","""79818""","""141""","""77"""
"""vourles plaine""","""117""","""bus""","""81943_48115""","""81943_117""","""81943_1""",45.65738,4.76232,"""Vourles La Plaine""","""Gare D’Oullins - St-Genis Lycé…","""TCL""","""FR""",2,false,false,"""2026-10-30""","""81943""","""81943_48115""","""81943_117"""


## 3. Le rapport par réseau

Il nécessite d'avoir lancé `run_gtfs_pipeline` pour la même date et le même dossier.

In [5]:
report = build_network_report(RUN_DATE, DATA_DIR)

print(report.shape)
report.head()

(563, 30)


dataset_id,resource_transport_id,resource_datagouv_id,dataset_title,resource_title,page_url,resource_url,resource_original_url,resource_page_url,publisher_name,resource_community_resource_publisher,resource_features,resource_modes,stops_count,categorie_geo,covered_area,is_scolaire,is_saisonnier,is_agregat,offer_ids,offer_names,legal_owners,created_at,updated,resource_extract_date,agency_name,start_date,end_date,agregat_resources_id,service_dates_valid
str,i64,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,bool,bool,bool,str,str,str,str,str,str,str,str,str,i64,bool
"""6703a201386f775e01f4269b""",82386,"""eae0fa46-087a-4018-ada9-d8add1…","""Réseau européen AVE Renfe""","""Horaires AVE Espagne -France""","""https://transport.data.gouv.fr…","""https://www.data.gouv.fr/api/1…","""https://ssl.renfe.com/gtransit…","""https://transport.data.gouv.fr…","""RENFE VIAJEROS SOCIEDAD MERCAN…",null,"""position des stations,horaires…","""rail""",16,"""pays""","""{""type"":""pays"",""nom"":""France"",…",false,false,false,"""""","""""","""{""insee"":null,""name"":null,""typ…","""2024-10-07""","""2026-10-29T16:58:49.000000Z""","""2026-10-30""","""RENFE OPERADORA""","""29/10/2026""","""24/01/2027""",null,true
"""5cdef3698b4c416d21fd76b9""",52605,"""fd54f81f-4389-4e73-be75-491133…","""Réseau européen BlaBlaCar Bus""","""GTFS BlaBlaCar Bus""","""https://transport.data.gouv.fr…","""https://www.data.gouv.fr/api/1…","""https://drive.google.com/uc?ex…","""https://transport.data.gouv.fr…","""BlaBlaCar Bus""",null,"""position des stations,horaires…","""bus""",660,"""pays""","""{""type"":""pays"",""nom"":""France"",…",false,false,false,"""""","""""","""{""insee"":null,""name"":null,""typ…","""2019-05-17""","""2026-10-30T03:47:02.830521Z""","""2026-10-30""","""BlaBlaCar Bus""","""30/10/2026""","""06/01/2027""",null,true
"""63bc5f36f25deab1e855c0ff""",82199,"""bfd97acd-63f3-4ea4-bfe8-70e4c7…","""Réseau européen Eurostar""","""Eurostar GTFS a partir du 16/0…","""https://transport.data.gouv.fr…","""https://www.data.gouv.fr/api/1…","""https://integration-storage.dm…","""https://transport.data.gouv.fr…","""Eurostar International Ltd.""",null,"""position des stations,horaires…","""rail""",173,"""pays""","""{""type"":""pays"",""nom"":""France"",…",false,false,false,"""""","""""","""{""insee"":null,""name"":null,""typ…","""2023-01-09""","""2026-10-29T16:00:52.000000Z""","""2026-10-30""","""Eurostar""","""30/10/2026""","""27/01/2027""",null,true
"""5c6ad5248b4c411c3d7ae435""",11681,"""30d94e83-48a4-4c44-8a96-c08237…","""Réseau européen FlixBus et Fli…","""Horaires Flixbus Europe""","""https://transport.data.gouv.fr…","""https://www.data.gouv.fr/api/1…","""https://gtfs.gis.flix.tech/gtf…","""https://transport.data.gouv.fr…","""Flixbus France""",null,"""position des stations,horaires…","""bus,rail""",2116,"""pays""","""{""type"":""pays"",""nom"":""France"",…",false,false,false,"""""","""""","""{""insee"":null,""name"":null,""typ…","""2019-02-18""","""2026-10-29T15:21:08.691821Z""","""2026-10-30""","""FlixBus-eu, FlixTrain-eu""","""25/10/2026""","""25/04/2027""",null,true
"""64635525318cc75a9a8a771f""",81653,"""bdecea2c-ebc9-4f22-812d-927e4a…","""Réseau national Trenitalia Fra…","""GTFS Trenitalia France""","""https://transport.data.gouv.fr…","""https://www.data.gouv.fr/api/1…","""https://thello.axelor.com/publ…","""https://transport.data.gouv.fr…","""TRENITALIA FRANCE""",null,"""position des stations,horaires…","""rail""",16,"""pays""","""{""type"":""pays"",""nom"":""France"",…",false,false,false,"""""","""""","""{""insee"":null,""name"":null,""typ…","""2023-05-16""","""2026-10-29T16:05:05.000000Z""","""2026-10-30""","""Trenitalia""","""29/10/2026""","""11/12/2026""",null,true
